# 02. 유한 FIFO와 패킷 이벤트
목표: 논문 §2의 생성·전송·전파·큐·드롭을 분리하고 완료 패킷만 평가할 때의 편향을 확인한다. Python 3.10+ 표준 라이브러리만 사용한다. 모든 셀을 순서대로 실행한다. [원문](https://doi.org/10.5281/zenodo.13885645).

Toy reproduction: heapq 기반 단일 링크. SimPy, 위성 궤도, SNR, DDQN은 구현하지 않는다. 패킷 길이는 일정하다. capacity는 **서비스 중인 패킷을 포함한 송신 시스템 용량**이며 전파 중 패킷은 포함하지 않는다. 도착 생성은 horizon에서 멈추고 drain=True이면 수용한 패킷은 끝까지 처리한다.

In [ ]:
import heapq, math, random, statistics
from collections import deque
def arrivals_poisson(rate, horizon, seed):
    if not all(math.isfinite(x) and x > 0 for x in (rate, horizon)):
        raise ValueError('positive rate and horizon required')
    rng, time, result = random.Random(seed), 0.0, []
    while True:
        time += rng.expovariate(rate)  # Poisson 과정의 도착 간격은 지수분포
        if time >= horizon:
            return result
        result.append(time)
def simulate(arrivals, horizon=1.0, capacity=8, service=0.001, propagation=0.003, drain=True):
    if not all(math.isfinite(x) for x in (service, propagation, horizon)):
        raise ValueError('finite configuration required')
    if not isinstance(capacity, int) or capacity < 1 or service <= 0 or propagation < 0 or horizon <= 0:
        raise ValueError('invalid link configuration')
    if arrivals != sorted(arrivals) or any(not math.isfinite(t) or t < 0 or t >= horizon for t in arrivals):
        raise ValueError('arrivals must be ordered inside horizon')
    events, waiting, records = [], deque(), []
    busy = False
    accepted = dropped = 0
    # 동시각이면 송신 완료(0), 새 도착(1), 수신 완료(2) 순서로 처리한다.
    for pid, time in enumerate(arrivals):
        heapq.heappush(events, (time, 1, pid, time, 0.0))
    while events:
        now, kind, pid, born, queue_time = heapq.heappop(events)
        if not drain and now > horizon:
            break
        if kind == 1:
            if len(waiting) + int(busy) >= capacity:
                dropped += 1
                continue
            accepted += 1
            if busy:
                waiting.append((pid, born))
            else:
                busy = True
                heapq.heappush(events, (now + service, 0, pid, born, 0.0))
        elif kind == 0:
            # 전파를 기다리는 동안에도 다음 패킷의 송신을 시작할 수 있다.
            heapq.heappush(events, (now + propagation, 2, pid, born, queue_time))
            if waiting:
                nxt, arrived = waiting.popleft()
                heapq.heappush(events, (now + service, 0, nxt, arrived, now - arrived))
            else:
                busy = False
        else:
            records.append({'id': pid, 'queue': queue_time, 'latency': now - born})
    assert accepted + dropped == len(arrivals)
    for row in records:
        assert row['queue'] >= 0
        assert math.isclose(row['latency'], row['queue'] + service + propagation, abs_tol=1e-10)
    delivered = len(records)
    if drain:
        assert delivered == accepted
    return {'generated': len(arrivals), 'accepted': accepted, 'dropped': dropped,
            'delivered': delivered, 'unfinished': accepted - delivered, 'records': records}
def summary(result):
    delays = sorted(r['latency'] for r in result['records'])
    return {**{k:v for k,v in result.items() if k != 'records'},
            'mean_ms': round(statistics.mean(delays)*1000, 4) if delays else None,
            'p95_ms': round(delays[math.ceil(.95*len(delays))-1]*1000, 4) if delays else None}


In [ ]:
# 손으로 계산할 수 있는 사례로 사건 순서와 용량의 의미부터 확인한다.
r = simulate([0.0, 0.0, 0.0], capacity=2)
assert (r['accepted'], r['dropped'], r['delivered']) == (2, 1, 2)
assert [round(x['latency'], 6) for x in r['records']] == [0.004, 0.005]
assert [x['id'] for x in r['records']] == [0, 1]
assert simulate([], capacity=1)['delivered'] == 0
r = simulate([0.0, 0.001], capacity=1)
assert r['dropped'] == 0  # 완료와 같은 시각의 도착은 빈 슬롯을 사용한다.
late = [0.999]
assert simulate(late, drain=False)['unfinished'] == 1
assert simulate(late, drain=True)['unfinished'] == 0
try:
    simulate([0.0], capacity=0)
    raise AssertionError('zero capacity accepted')
except ValueError:
    pass
print('FIFO, capacity, simultaneous event, empty input, horizon tests passed')


In [ ]:
for rate in (300, 1500):
    # 서비스 1ms이므로 rho = lambda * service; 1을 넘으면 offered load가 용량 초과
    reports = [summary(simulate(arrivals_poisson(rate, 1.0, seed))) for seed in range(5)]
    print('lambda=', rate, 'rho=', rate * .001)
    for report in reports:
        print(report)
trace = arrivals_poisson(1500, 1.0, 42)
print('same trace, drain=False:', summary(simulate(trace, drain=False)))
print('same trace, drain=True :', summary(simulate(trace, drain=True)))
assert simulate(trace) == simulate(trace)  # 동일 입력의 재현성을 확인한다.


## 해석·확장 과제
높은 rho에서는 유한 버퍼 때문에 지연이 무한히 커지는 대신 드롭이 증가할 수 있다. 낮아진 평균 지연만으로 정책이 좋아졌다고 판단하지 않는다. p95는 nearest-rank 정의다. 통계는 완료된 패킷의 값이므로 반드시 unfinished와 함께 읽는다.

과제: capacity 1/8/32를 같은 도착 trace로 비교하라. propagation을 10배로 바꿔도 송신 큐의 처리율은 바뀌지 않아야 한다. 링크별 큐가 있는 다중 홉 망으로 확장할 때 도착 이벤트가 다음 링크의 큐로 이어지게 하라. 이 notebook의 단일 링크 수치는 논문의 군집 실험과 직접 비교할 수 없다.